# 01 PDF Parser -- Text and Image Extraction from Immunology Textbooks

## Purpose

This notebook walks through the **first step** of the ImmunoBiology RAG pipeline: extracting structured text and images from PDF files (primarily Janeway's Immunobiology 10th Edition).

## Position in Pipeline

```
[01_pdf_parser] --> 02_chunker --> 03_embedder --> 04_retriever
```

PDF parsing is the foundation of the entire system. Every downstream component (chunking, embedding, retrieval) depends on the quality of text extraction performed here.

## Learning Objectives

1. Understand how PyMuPDF (fitz) extracts text blocks and images from PDF pages
2. Learn how layout inspection guides parameter calibration (header/footer crop, image thresholds)
3. See how chapter headings and figure captions are detected using regex patterns
4. Examine the metadata schema attached to each extracted page
5. Visualize extraction quality with page text length distributions

## Imports

We use `fitz` (PyMuPDF) for PDF manipulation, `json` for serialization, `Path` for cross-platform file handling, and `matplotlib` for visualization.

In [ ]:
import sys
import json
from pathlib import Path
from glob import glob

import fitz  # PyMuPDF
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

# Add project root to sys.path so we can import from src/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"PyMuPDF version: {fitz.__doc__}")

## Configuration

All paths and parameters are centralized in `src/constant.py`, which loads from `config.yaml`. This avoids hardcoding paths in notebooks.

In [ ]:
from src import constant

# Key paths from config.yaml
RAW_DIR = Path(constant.raw_dir)
PROCESSED_DIR = Path(constant.processed_dir)
DIAGNOSTICS_DIR = Path(constant.diagnostics_dir)

print(f"Raw PDF directory:   {RAW_DIR}")
print(f"Processed output:    {PROCESSED_DIR}")
print(f"Diagnostics output:  {DIAGNOSTICS_DIR}")

# Discover available PDFs
pdf_files = sorted(RAW_DIR.glob("*.pdf"))
print(f"\nFound {len(pdf_files)} PDF(s):")
for p in pdf_files:
    print(f"  - {p.name} ({p.stat().st_size / 1024 / 1024:.1f} MB)")

## Layout Inspection

Before parsing, we inspect the PDF layout to calibrate key parameters:
- **Header/footer clip**: how many points to crop from the top/bottom (to remove running titles and page numbers)
- **Image thresholds**: minimum width/height/size to keep (to filter out decorative icons)
- **Font sizes**: identify body text vs. heading fonts

The `inspect_pdf_layout()` function samples a few pages and reports these statistics. Always run this first when working with a new PDF.

In [ ]:
from src.pdf_parser import inspect_pdf_layout

# Inspect the first available PDF
if pdf_files:
    target_pdf = pdf_files[0]
    print(f"Inspecting: {target_pdf.name}\n")
    layout_report = inspect_pdf_layout(target_pdf, sample_pages=[0, 49, 99, 199])
    
    print(f"\n--- Summary ---")
    print(f"Total pages: {layout_report['total_pages']}")
    if layout_report['font_sizes']:
        fs = layout_report['font_sizes']
        print(f"Font sizes: min={min(fs):.1f}, median={sorted(fs)[len(fs)//2]:.1f}, max={max(fs):.1f}")
    if layout_report['image_widths']:
        print(f"Images found: {len(layout_report['image_widths'])}")
else:
    print("No PDF files found in data/raw/. Place your PDF there and re-run.")

## Parse a Single Page

Let us open a PDF and examine a single page in detail. We will:
1. Load the page and get its text blocks
2. Show the raw text with header/footer cropping applied
3. Demonstrate chapter heading detection

This gives intuition for what the full parser does on every page.

In [ ]:
from src.pdf_parser import (
    PAGE_HEADER_CLIP, PAGE_FOOTER_CLIP, MIN_CONTENT_PAGE,
    detect_chapter, detect_layout, CHAPTER_HEADING_PATTERN
)

if pdf_files:
    pdf = fitz.open(str(pdf_files[0]))
    # Pick a content page (after front matter)
    sample_page_idx = min(MIN_CONTENT_PAGE + 5, len(pdf) - 1)
    page = pdf.load_page(sample_page_idx)
    
    print(f"Page {sample_page_idx + 1} (0-indexed: {sample_page_idx})")
    print(f"Dimensions: {page.rect.width:.1f} x {page.rect.height:.1f} pt")
    print(f"Layout type: {detect_layout(page)}")
    
    # Apply header/footer cropping
    crop = fitz.Rect(0, PAGE_HEADER_CLIP, page.rect.width, page.rect.height - PAGE_FOOTER_CLIP)
    text = page.get_text(clip=crop).strip()
    
    print(f"\nText length: {len(text)} characters")
    print(f"Text blocks on page: {len(page.get_text('blocks'))}")
    
    # Chapter detection
    chapter = detect_chapter(text, "Chapter 1")
    print(f"Detected chapter: {chapter}")
    
    # Show first 500 characters
    print(f"\n--- Page text (first 500 chars) ---")
    print(text[:500])
    print("...")
    
    pdf.close()
else:
    print("No PDF files available.")

## Image Extraction

The parser extracts images that meet minimum size thresholds (100x100 px, 5 KB). Small icons and decorative elements are filtered out. For each qualifying image, the system:
1. Saves the image to `data/processed/<doc>/images/`
2. Searches for nearby text blocks that match figure caption patterns
3. Records the caption, page number, and source file in metadata

Below we demonstrate extracting images from a single page.

In [ ]:
from src.pdf_parser import handle_image, IMAGE_MIN_WIDTH, IMAGE_MIN_HEIGHT, IMAGE_MIN_BYTES

print(f"Image extraction thresholds:")
print(f"  Min width:  {IMAGE_MIN_WIDTH} px")
print(f"  Min height: {IMAGE_MIN_HEIGHT} px")
print(f"  Min size:   {IMAGE_MIN_BYTES} bytes")

if pdf_files:
    pdf = fitz.open(str(pdf_files[0]))
    
    # Find a page with images
    found_images = False
    for page_idx in range(MIN_CONTENT_PAGE, min(MIN_CONTENT_PAGE + 50, len(pdf))):
        page = pdf.load_page(page_idx)
        images = page.get_images(full=True)
        if images:
            print(f"\nPage {page_idx + 1}: found {len(images)} image(s)")
            for i, img in enumerate(images):
                xref = img[0]
                base = pdf.extract_image(xref)
                w, h = base['width'], base['height']
                sz = len(base['image'])
                status = "KEEP" if (w >= IMAGE_MIN_WIDTH and h >= IMAGE_MIN_HEIGHT and sz >= IMAGE_MIN_BYTES) else "SKIP"
                print(f"  [{status}] Image {i+1}: {base['ext']} {w}x{h} px, {sz/1024:.1f} KB")
            found_images = True
            break
    
    if not found_images:
        print("No images found in the sampled page range.")
    
    pdf.close()

## Full PDF Parse

The `load_all_pdfs()` function discovers all PDFs in `data/raw/`, parses each one page by page, and produces:
- A list of LangChain `Document` objects (one per content page)
- An `extraction_report.json` summarizing pages, images, and chapters per document

Each Document carries rich metadata: `unique_id`, `source_file`, `doc_type`, `chapter`, `page`, `has_figure_caption`, and `images_info`.

**Note**: This cell may take several minutes for large textbooks.

In [ ]:
from src.pdf_parser import load_all_pdfs

# Parse all PDFs (this is the primary entry point used by build_index.py)
all_docs, extraction_report = load_all_pdfs()

print(f"\nTotal pages extracted: {len(all_docs)}")
print(f"\n--- Extraction Report ---")
print(json.dumps(extraction_report, indent=2, ensure_ascii=False))

# Show sample document metadata
if all_docs:
    sample = all_docs[0]
    print(f"\n--- Sample Document (page {sample.metadata['page']}) ---")
    print(f"source_file: {sample.metadata['source_file']}")
    print(f"doc_type:    {sample.metadata['doc_type']}")
    print(f"chapter:     {sample.metadata['chapter']}")
    print(f"unique_id:   {sample.metadata['unique_id'][:16]}...")
    print(f"images:      {len(sample.metadata['images_info'])}")
    print(f"text length: {len(sample.page_content)} chars")

## Visualization: Page Text Length Distribution

A histogram of text length per page helps us verify the extraction quality:
- Very short pages may indicate failed extraction or image-heavy pages
- Very long pages may indicate OCR artifacts or multiple columns merged
- The bulk should fall within a reasonable range for the textbook format

In [ ]:
if all_docs:
    page_lengths = [len(doc.page_content) for doc in all_docs]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(page_lengths, bins=50, edgecolor="black", color="steelblue", alpha=0.8)
    axes[0].set_title("Page Text Length Distribution", fontsize=13)
    axes[0].set_xlabel("Characters per page")
    axes[0].set_ylabel("Count")
    axes[0].axvline(sum(page_lengths) / len(page_lengths), color="red", linestyle="--",
                    label=f"Mean: {sum(page_lengths)/len(page_lengths):.0f}")
    axes[0].legend()
    
    # Statistics annotation
    stats = (
        f"n={len(page_lengths)}\n"
        f"min={min(page_lengths)}\n"
        f"median={sorted(page_lengths)[len(page_lengths)//2]}\n"
        f"max={max(page_lengths)}\n"
        f"mean={sum(page_lengths)/len(page_lengths):.0f}"
    )
    axes[0].text(0.98, 0.97, stats, transform=axes[0].transAxes,
                 verticalalignment="top", horizontalalignment="right",
                 fontsize=9, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
    
    # Cumulative distribution
    sorted_lens = sorted(page_lengths)
    cumulative = [i / len(sorted_lens) for i in range(len(sorted_lens))]
    axes[1].plot(sorted_lens, cumulative, color="steelblue")
    axes[1].set_title("Cumulative Distribution", fontsize=13)
    axes[1].set_xlabel("Characters per page")
    axes[1].set_ylabel("Cumulative fraction")
    
    plt.tight_layout()
    plt.show()
    
    # Pages with very short text (potential issues)
    short_pages = [doc for doc in all_docs if len(doc.page_content) < 200]
    if short_pages:
        print(f"\nWarning: {len(short_pages)} pages have fewer than 200 characters.")
        for doc in short_pages[:5]:
            print(f"  Page {doc.metadata['page']}: {len(doc.page_content)} chars")
else:
    print("No documents to visualize. Run the full parse cell above first.")

## Summary

### Outputs Produced

| Output | Location | Description |
|--------|----------|-------------|
| Page JSON files | `data/processed/<doc>/text/page_NNNN.json` | One JSON per page with text + metadata |
| Extracted images | `data/processed/<doc>/images/` | Filtered images (>100x100 px, >5 KB) |
| Layout report | `outputs/diagnostics/pdf_layout_report.txt` | PDF structure analysis |
| Extraction report | `data/processed/extraction_report.json` | Per-document extraction statistics |
| LangChain Documents | In-memory list (`all_docs`) | Ready for chunking in the next step |

### Next Notebook

Proceed to **`02_chunker.ipynb`** to split extracted pages into semantic parent-child chunks for retrieval.